# 04 — Un réseau de neurones à l'œuvre : l'autoencodeur

**IA-01 · Jour 4 · Hexalp**

Hugo a entraîné un **autoencodeur** : un petit réseau de neurones qui a appris à **recopier** les connexions normales de septembre, en les faisant passer par un goulot de 3 neurones. Ce qu'il a l'habitude de voir, il le recopie bien. Ce qui est inhabituel, il le recopie mal : son **erreur de reconstruction** devient un score d'anomalie.

Vous n'entraînez rien : vous chargez le modèle de Hugo et vous l'utilisez. C'est ce que vous ferez le plus souvent avec un modèle de Deep Learning.

```
 7 caractéristiques → 8 neurones → 3 neurones → 8 neurones → 7 caractéristiques reconstruites
     (entrée)                       (goulot)                          (sortie)
```

---
## 1. Vérifier avant de charger

Un modèle Scikit-learn s'enregistre avec **joblib**, qui s'appuie sur le format **pickle** de Python. Charger un tel fichier, c'est **exécuter du code** : un fichier piégé peut faire n'importe quoi sur votre machine. On vérifie donc son empreinte SHA-256 avant de l'ouvrir. L'empreinte attendue vous a été transmise par un autre canal que le fichier : elle figure dans le kit, séquence 4.7.

**Exécution locale avec assistance, hors VM.** Référence SHA-256 relevée dans le PDF du cours (4.7). Cette concordance vérifie la conformité au kit ; elle ne certifie pas la provenance réelle de Hugo ni l’innocuité du contenu. Comparaison principale : préparation pédagogique avec médiane et IF0,05 ; V2 à exclusion/0,03 distincte.

In [1]:
import hashlib

EMPREINTE_ATTENDUE = "01464c560c949742a4b832c1b615194971ce56e29c93f39fd98bda3169e38b00"

with open("autoencodeur_hexalp.joblib", "rb") as f:
    empreinte = hashlib.sha256(f.read()).hexdigest()

print("Empreinte du fichier :", empreinte)
if empreinte != EMPREINTE_ATTENDUE:
    raise ValueError("Empreinte différente : ne chargez pas ce fichier, prévenez le SOC.")
print("✅ Empreinte conforme")

Empreinte du fichier : 01464c560c949742a4b832c1b615194971ce56e29c93f39fd98bda3169e38b00
✅ Empreinte conforme


In [2]:
from pathlib import Path
if hashlib.sha256(Path("autoencodeur_hexalp.joblib").read_bytes()).hexdigest() != EMPREINTE_ATTENDUE:
    raise ValueError("Empreinte différente : aucun chargement")
import joblib
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import IsolationForest

modele = joblib.load("autoencodeur_hexalp.joblib")
print(modele["description"])

normaliseur = modele["normaliseur"]
reseau = modele["reseau"]
SEUIL = modele["seuil"]
CARACTERISTIQUES = modele["caracteristiques"]

Autoencodeur Hexalp — entraîné le 30/09/2026 par Hugo (SOC) sur les 1 587 connexions SSH de septembre étiquetées normales. Seuil : 95e centile de l'erreur de reconstruction sur septembre.


---
## 2. Ouvrir la boîte

Le réseau est fait de couches de neurones. Chaque connexion entre deux neurones porte un **poids**, et chaque neurone un **biais** : ce sont les paramètres trouvés pendant l'entraînement.

In [3]:
print("Neurones par couche :", [reseau.n_features_in_] + list(reseau.hidden_layer_sizes) + [reseau.n_outputs_])

nb_parametres = sum(poids.size for poids in reseau.coefs_) + sum(biais.size for biais in reseau.intercepts_)
print("Nombre de paramètres :", nb_parametres)

Neurones par couche : [7, 8, 3, 8, 7]
Nombre de paramètres : 186


In [4]:
# Les 56 poids entre l'entrée et la première couche cachée
pd.DataFrame(reseau.coefs_[0], index=CARACTERISTIQUES, columns=[f"neurone {i+1}" for i in range(8)]).round(2)

,neurone 1,neurone 2,neurone 3,neurone 4,neurone 5,neurone 6,neurone 7,neurone 8
echecs_auth,0.04,-0.03,-0.22,-0.07,-0.08,-0.19,0.01,0.12
succes_auth,0.07,0.07,-0.07,0.03,0.16,0.09,-0.20,-0.01
comptes_distincts,0.06,-0.07,-0.20,0.01,0.01,-0.19,-0.01,0.11
duree_session_min,0.11,-0.02,-0.31,-0.24,0.05,-0.14,-0.22,-0.26
octets_sortants_mo,-0.04,-0.09,0.12,0.13,0.13,-0.03,0.16,-0.02
heure,0.14,-0.04,-0.03,0.06,0.09,0.02,-0.27,-0.10
externe,0.34,0.07,0.77,0.01,1.07,0.40,-0.23,0.40


✍️ **R1.** Combien de paramètres compte ce réseau ? Combien de seuils comptait votre arbre de décision ? Regardez le tableau des poids : pouvez-vous dire, comme avec l'arbre, « le réseau alerte quand… » ?

---
## 3. Utiliser le modèle sur octobre

In [5]:
octobre = pd.read_csv("hexalp_ssh_oct2026.csv")
octobre["octets_sortants_mo"] = octobre["octets_sortants_mo"].fillna(octobre["octets_sortants_ko"] / 1024)
octobre["octets_sortants_mo"] = octobre["octets_sortants_mo"].fillna(octobre["octets_sortants_mo"].median())
octobre["externe"] = (octobre["origine"] == "externe").astype(int)

X = octobre[CARACTERISTIQUES]
Z = normaliseur.transform(X)                      # mise à l'échelle, comme pendant l'entraînement
reconstruction = reseau.predict(Z)                # le réseau tente de recopier chaque ligne
erreurs = (Z - reconstruction) ** 2               # l'écart, caractéristique par caractéristique

octobre["erreur"] = erreurs.mean(axis=1)
octobre["anomalie_ae"] = (octobre["erreur"] > SEUIL).astype(int)

pd.crosstab(octobre["anomalie_ae"], octobre["etiquette"])

etiquette,attaque,normal
anomalie_ae,,
0,145,1898
1,70,6


In [6]:
# Pour comparer : l'Isolation Forest du J3, entraîné sur septembre
septembre = pd.read_csv("hexalp_ssh_sept2026.csv")
septembre["octets_sortants_mo"] = septembre["octets_sortants_mo"].fillna(septembre["octets_sortants_mo"].median())
septembre["externe"] = (septembre["origine"] == "externe").astype(int)
iforest = IsolationForest(contamination=0.05, random_state=42).fit(septembre[CARACTERISTIQUES])
octobre["anomalie_if"] = (iforest.predict(X) == -1).astype(int)

groupes = {
    "Sauvegarde nocturne (normal)": octobre["ip_source"] == "10.20.8.30",
    "Proxy Zabbix (normal)": octobre["ip_source"] == "10.20.1.7",
    "Exfiltrations par VPN (attaque)": (octobre["etiquette"] == "attaque") & octobre["ip_source"].str.startswith("10.20.250."),
    "Brute force (attaque)": (octobre["etiquette"] == "attaque") & (octobre["echecs_auth"] >= 40) & (octobre["succes_auth"] == 0),
    "Toutes les attaques": octobre["etiquette"] == "attaque",
    "Toutes les lignes normales": octobre["etiquette"] == "normal",
}
pd.DataFrame({nom: {"lignes": masque.sum(),
                    "signalées par l'Isolation Forest": octobre.loc[masque, "anomalie_if"].sum(),
                    "signalées par l'autoencodeur": octobre.loc[masque, "anomalie_ae"].sum()}
              for nom, masque in groupes.items()}).T

,lignes,signalées par l'Isolation Forest,signalées par l'autoencodeur
Sauvegarde nocturne (normal),28,19,4
Proxy Zabbix (normal),278,0,0
Exfiltrations par VPN (attaque),9,9,9
Brute force (attaque),50,40,50
Toutes les attaques,215,61,70
Toutes les lignes normales,1904,22,6


✍️ **R2.** Où l'autoencodeur fait-il mieux que l'Isolation Forest ? Où fait-il moins bien ? Rappelez-vous sur quelles données chacun a été entraîné.

---
## 4. Expliquer une alerte

Le réseau ne donne pas de règle. Mais on peut regarder **quelle caractéristique il a le plus de mal à recopier**.

In [7]:
def detail(ligne):
    i = octobre.index.get_loc(ligne)
    return pd.DataFrame({"valeur": X.iloc[i], "erreur de reconstruction": erreurs[i].round(2)})

# L'exfiltration du 9 octobre à 3 h, repérée à la main par Hugo
exfiltration = octobre[(octobre["date"] == "2026-10-09") & (octobre["heure"] == 3) & octobre["ip_source"].str.startswith("10.20.250.")].index[0]
detail(exfiltration)

,valeur,erreur de reconstruction
echecs_auth,1.000000,0.26
succes_auth,1.000000,0.08
comptes_distincts,1.000000,0.60
duree_session_min,50.000000,0.52
octets_sortants_mo,8690.075195,928.97
heure,3.000000,0.27
externe,0.000000,0.08


In [8]:
# Une nuit de sauvegarde ordinaire
sauvegarde = octobre[octobre["ip_source"] == "10.20.8.30"].index[0]
detail(sauvegarde)

,valeur,erreur de reconstruction
echecs_auth,0.00,0.00
succes_auth,2.00,0.04
comptes_distincts,1.00,0.01
duree_session_min,51.00,0.00
octets_sortants_mo,1666.53,0.15
heure,1.00,0.03
externe,0.00,0.00


✍️ **R3.** Qu'est-ce qui « dérange » le réseau dans l'exfiltration du 9 octobre ? Cette explication suffirait-elle à Nadia pour décider de couper un accès VPN ?

✍️ **R4.** Ce fichier a été entraîné par Hugo, sur un poste du SOC, puis copié sur votre machine. Citez trois façons dont il aurait pu être modifié en chemin. Que vérifie l'empreinte SHA-256 ? Que ne vérifie-t-elle pas ?

✍️ **R5.** Les grands modèles que vous utilisez tous les jours (J1) sont des réseaux du même type, avec des milliards de paramètres au lieu de 186. Que retenez-vous de cette séquence pour expliquer leurs décisions ou vérifier les fichiers que vous téléchargez ?

---
✅ **Fin du notebook.**

## Preuves complémentaires et comparaison au périmètre V2
Inférence du même autoencodeur sans réentraînement ; exclusion des 14 vrais volumes absents pour comparer sur les 2 105 mêmes observations que la V2.

In [9]:
import json, sys, sklearn, numpy as np
from sklearn.metrics import confusion_matrix
from pathlib import Path
import hashlib
raw = pd.read_csv("hexalp_ssh_oct2026.csv")
volume = raw["octets_sortants_mo"].fillna(raw["octets_sortants_ko"] / 1024)
couverture = volume.notna()
def bilan(frame, col):
    y = (frame["etiquette"] == "attaque").astype(int)
    tn, fp, fn, tp = confusion_matrix(y, frame[col], labels=[0,1]).ravel()
    return {"VN":int(tn),"FP":int(fp),"FN":int(fn),"VP":int(tp),
            "precision":float(tp/(tp+fp)) if tp+fp else None,
            "rappel":float(tp/(tp+fn)), "alertes":int(tp+fp)}
groupes_stats = {nom:{"lignes":int(m.sum()),"IF":int(octobre.loc[m,"anomalie_if"].sum()),
    "AE":int(octobre.loc[m,"anomalie_ae"].sum())} for nom,m in groupes.items()}
resultat = {"origine":"Assistant, inférence locale hors VM", "python":sys.version.split()[0],
    "scikit_learn":sklearn.__version__, "pandas":pd.__version__,
    "empreinte_reference_PDF":EMPREINTE_ATTENDUE,
    "empreinte_apres_inference":hashlib.sha256(Path("autoencodeur_hexalp.joblib").read_bytes()).hexdigest(),
    "couches":[int(reseau.n_features_in_)]+list(reseau.hidden_layer_sizes)+[int(reseau.n_outputs_)],
    "parametres":int(nb_parametres),"seuil":float(SEUIL), "caracteristiques":list(CARACTERISTIQUES),
    "description":modele["description"],
    "bilan_AE_2119":bilan(octobre,"anomalie_ae"), "bilan_IF_2119":bilan(octobre,"anomalie_if"),
    "bilan_AE_2105":bilan(octobre.loc[couverture],"anomalie_ae"),"groupes_2119":groupes_stats,
    "attaques_IF_seul":int(((octobre.etiquette == "attaque") & (octobre.anomalie_if == 1) & (octobre.anomalie_ae == 0)).sum()),
    "attaques_AE_seul":int(((octobre.etiquette == "attaque") & (octobre.anomalie_ae == 1) & (octobre.anomalie_if == 0)).sum()),
    "detail_exfiltration":detail(exfiltration).to_dict(),
    "detail_premiere_sauvegarde":detail(sauvegarde).to_dict(),
    "date_premiere_sauvegarde":str(octobre.loc[sauvegarde,"date"]),
    "heure_premiere_sauvegarde":int(octobre.loc[sauvegarde,"heure"])}
Path("preuves/autoencodeur_47.json").write_text(json.dumps(resultat,ensure_ascii=False,indent=2)+"\n")
print(json.dumps(resultat,ensure_ascii=False,indent=2))


{
  "origine": "Assistant, inférence locale hors VM",
  "python": "3.14.4",
  "scikit_learn": "1.9.1",
  "pandas": "3.0.6",
  "empreinte_reference_PDF": "01464c560c949742a4b832c1b615194971ce56e29c93f39fd98bda3169e38b00",
  "empreinte_apres_inference": "01464c560c949742a4b832c1b615194971ce56e29c93f39fd98bda3169e38b00",
  "couches": [
    7,
    8,
    3,
    8,
    7
  ],
  "parametres": 186,
  "seuil": 2.7615,
  "caracteristiques": [
    "echecs_auth",
    "succes_auth",
    "comptes_distincts",
    "duree_session_min",
    "octets_sortants_mo",
    "heure",
    "externe"
  ],
  "description": "Autoencodeur Hexalp — entraîné le 30/09/2026 par Hugo (SOC) sur les 1 587 connexions SSH de septembre étiquetées normales. Seuil : 95e centile de l'erreur de reconstruction sur septembre.",
  "bilan_AE_2119": {
    "VN": 1898,
    "FP": 6,
    "FN": 145,
    "VP": 70,
    "precision": 0.9210526315789473,
    "rappel": 0.32558139534883723,
    "alertes": 76
  },
  "bilan_IF_2119": {
    "VN": 188